# Modelamiento — Punto 1: Clasificación para Focalización de Programas Sociales
**Modelación Taller 1 · HE2 · Consultoría Económica con IA Responsable**

| Sección | Contenido |
|---|---|
| 1 | Carga de datos |
| 2 | Train / Test split y estandarización |
| 3 | Funciones auxiliares |
| 4 | Regresión Logística — modelo base |
| 5 | Tuning con GridSearchCV (F₂) |
| 6 | Evaluación del modelo tuneado |
| 7 | Análisis de umbral — Logit |
| 8 | Random Forest — modelo base y tuning |
| 9 | XGBoost — modelo base y tuning |
| 10 | Comparación de modelos |
| 11 | Exportación del modelo |

In [54]:
import warnings
import os
import joblib
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score,
    fbeta_score, precision_score, recall_score,
    roc_auc_score, make_scorer,
)

try:
    from xgboost import XGBClassifier
except ImportError:
    raise ImportError("XGBoost no encontrado. Instalar con:  pip install xgboost")

warnings.filterwarnings('ignore')

C_FN = 2_500 + 900 + 400   
C_FP = 1_200 + 150        

BETA_TEORICO = (C_FN / C_FP) ** 0.5
BETA         = 2
RANDOM_STATE = 42
TEST_SIZE    = 0.2
N_ITER       = 30
SCORING      = make_scorer(fbeta_score, beta=BETA, zero_division=0)

print('Configuración cargada:', {
    'C_FN': C_FN, 'C_FP': C_FP,
    'β_teórico': round(BETA_TEORICO, 3), 'β_usado': BETA,
    'RANDOM_STATE': RANDOM_STATE, 'TEST_SIZE': TEST_SIZE, 'N_ITER': N_ITER,
})

Configuración cargada: {'C_FN': 3800, 'C_FP': 1350, 'β_teórico': 1.678, 'β_usado': 2, 'RANDOM_STATE': 42, 'TEST_SIZE': 0.2, 'N_ITER': 30}


## 1. Carga de datos

El dataset fue generado por `preprocesamiento_punto1.py` y almacenado en `Datos/train_cleaned_hogar.csv`.
Ya incluye: imputación, limpieza, agregación al nivel de hogar y feature engineering (4 índices compuestos).
**Los datos NO están estandarizados** — el scaler se entrena solo sobre X_train en la sección 2. **Una fila = un hogar.**

In [55]:
DATA_PATH = os.path.join('Datos', 'train_cleaned_hogar.csv')

df = pd.read_csv(DATA_PATH)

n0   = (df['Target'] == 0).sum()
n1   = (df['Target'] == 1).sum()
prop = df['Target'].mean()

print(f'Shape: {df.shape[0]:,} hogares x {df.shape[1]} variables')
print(f'Target — 0 (No pobre): {n0:,}  |  1 (Pobre): {n1:,}')
print(f'Proporcion de hogares pobres: {prop:.3f}')

display(df.head(3))

Shape: 2,241 hogares x 52 variables
Target — 0 (No pobre): 1,727  |  1 (Pobre): 514
Proporcion de hogares pobres: 0.229


,idhogar,Monto_Alquiler,Promedio_Anos_Escolaridad,Pared_Zocalo,Pared_Prefabricado,Pared_Madera,Piso_Cemento,Piso_Madera,Techo_Zinc,Electricidad_Cooperativa,...,Zona_Urbana,Edad_Promedio,Hogar_con_Rezago_Escolar,Hogar_sin_Jefe_Educado,Tasa_Dependencia,Target,Indice_Activos_Tech,Privacion_Servicios_Basicos,Calidad_Materiales_Vivienda,Hacinamiento_Severo
0,001ff74ca,0.0,8.00,0,0,0,0,0,0,0,...,0,19.00,0,0,1.0,0,3.0,1,-3,0
1,003123ec2,0.0,3.25,0,1,0,1,0,1,0,...,1,12.75,0,0,1.0,1,3.0,0,0,0
2,004616164,0.0,7.00,0,0,1,0,1,1,0,...,0,33.00,1,0,1.0,1,3.0,3,0,0


## 2. Train / Test split y estandarización

Split 80/20.

In [56]:
from sklearn.preprocessing import StandardScaler

X = df.drop(columns=['Target', 'idhogar'], errors='ignore')
y = df['Target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f'X_train : {X_train.shape}  |  X_test : {X_test.shape}')
print(f'Balance train: {y_train.mean():.3f}  |  Balance test: {y_test.mean():.3f}')
print(f'Features del modelo: {X_train.shape[1]}')

# ── Estandarización: fit en X_train, transform en ambos ──────────────────
cols_continuas = [c for c in X_train.columns if X_train[c].nunique() > 2]

X_train = X_train.copy()
X_test  = X_test.copy()

scaler = StandardScaler()
X_train[cols_continuas] = scaler.fit_transform(X_train[cols_continuas])
X_test[cols_continuas]  = scaler.transform(X_test[cols_continuas])

print(f'\nVariables continuas estandarizadas : {len(cols_continuas)}')

# ── Guardar scaler ────────────────────────────────────────────────────────
scaler_dir  = 'Modelos'
scaler_path = os.path.join(scaler_dir, 'scaler.pkl')
os.makedirs(scaler_dir, exist_ok=True)
joblib.dump(scaler, scaler_path)
print(f'Scaler guardado : {scaler_path}')

X_train : (1792, 50)  |  X_test : (449, 50)
Balance train: 0.229  |  Balance test: 0.229
Features del modelo: 50

Variables continuas estandarizadas : 14
Scaler guardado : Modelos\scaler.pkl


## 3. Funciones auxiliares

In [57]:
def evaluate_model(y_true, y_pred, y_prob, beta=BETA):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        'accuracy':       accuracy_score(y_true, y_pred),
        'precision':      precision_score(y_true, y_pred, zero_division=0),
        'recall':         recall_score(y_true, y_pred, zero_division=0),
        'f1':             f1_score(y_true, y_pred, zero_division=0),
        f'f{beta}':       fbeta_score(y_true, y_pred, beta=beta, zero_division=0),
        'roc_auc':        roc_auc_score(y_true, y_prob),
        'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp,
        'cost_total_usd': C_FP * fp + C_FN * fn,
    }


def evaluate_thresholds(y_true, y_prob, thresholds, beta=BETA, c_fp=C_FP, c_fn=C_FN):
    rows = []
    for t in thresholds:
        y_pred_t = (y_prob >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred_t).ravel()
        rows.append({
            'threshold':      t,
            'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp,
            'precision':      precision_score(y_true, y_pred_t, zero_division=0),
            'recall':         recall_score(y_true, y_pred_t, zero_division=0),
            'f1':             f1_score(y_true, y_pred_t, zero_division=0),
            f'f{beta}':       fbeta_score(y_true, y_pred_t, beta=beta, zero_division=0),
            'cost_total_usd': c_fp * fp + c_fn * fn,
        })
    return pd.DataFrame(rows).sort_values('threshold').reset_index(drop=True)

## 4. Regresión Logística — modelo base

Modelo con hiperparámetros por defecto y umbral 0.5, para establecer una línea base.

In [58]:
logit_base = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
logit_base.fit(X_train, y_train)

y_prob_base = logit_base.predict_proba(X_test)[:, 1]
y_pred_base = (y_prob_base >= 0.5).astype(int)
metrics_base = evaluate_model(y_test, y_pred_base, y_prob_base)

cols_show = ['accuracy', 'precision', 'recall', 'f1', f'f{BETA}', 'roc_auc', 'cost_total_usd']
print('Métricas — Logit base (umbral 0.5):')
display(pd.DataFrame([metrics_base])[cols_show].round(4))

Métricas — Logit base (umbral 0.5):


,accuracy,precision,recall,f1,f2,roc_auc,cost_total_usd
0,0.7817,0.5424,0.3107,0.3951,0.3397,0.7786,306250


## 5. Tuning con GridSearchCV (F₂)

In [ ]:
param_grid = {
    'C':            [0.01, 0.1, 1.0, 10.0, 100.0],
    'penalty':      ['l1', 'l2'],
    'class_weight': [None, 'balanced'],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid = GridSearchCV(
    estimator=LogisticRegression(
        max_iter=2000,
        solver='liblinear',
        random_state=RANDOM_STATE,
    ),
    param_grid=param_grid,
    scoring=SCORING,
    cv=cv,
    n_jobs=-1,
    refit=True,
    verbose=0,
)

grid.fit(X_train, y_train)

print(f'Mejores hiperparametros : {grid.best_params_}')
print(f'Mejor F{BETA} en CV      : {grid.best_score_:.4f}')

Mejores hiperparametros : {'C': 0.01, 'class_weight': 'balanced', 'penalty': 'l2'}
Mejor F2 en CV      : 0.6932


## 6. Evaluación del modelo tuneado

In [60]:
best_logit   = grid.best_estimator_
y_prob_tuned = best_logit.predict_proba(X_test)[:, 1]
y_pred_tuned = (y_prob_tuned >= 0.5).astype(int)
metrics_tuned = evaluate_model(y_test, y_pred_tuned, y_prob_tuned)

comp_cols = ['accuracy', 'precision', 'recall', 'f1', f'f{BETA}', 'roc_auc', 'cost_total_usd']
comp = pd.DataFrame([
    {'modelo': 'base',  **{k: metrics_base[k]  for k in comp_cols}},
    {'modelo': 'tuned', **{k: metrics_tuned[k] for k in comp_cols}},
]).set_index('modelo')

print('Comparación base vs tuned (umbral 0.5):')
display(comp.round(4))

Comparación base vs tuned (umbral 0.5):


,accuracy,precision,recall,f1,f2,roc_auc,cost_total_usd
modelo,,,,,,,
base,0.7817,0.5424,0.3107,0.3951,0.3397,0.7786,306250
tuned,0.7149,0.4309,0.7573,0.5493,0.6577,0.7749,234050


## 7. Análisis de umbral — Logit

El umbral por defecto (0.5) no es necesariamente óptimo. Se evalúan umbrales en [0.15, 0.80] para encontrar:
1. El umbral que **maximiza F₂**
2. El umbral que **minimiza el costo total**

In [61]:
thresholds = np.arange(0.15, 0.80, 0.05).round(2)
thresh_df  = evaluate_thresholds(y_test, y_prob_tuned, thresholds)

best_f2   = thresh_df.loc[thresh_df[f'f{BETA}'].idxmax()]
best_cost = thresh_df.loc[thresh_df['cost_total_usd'].idxmin()]

cols_thresh = ['threshold', 'fp', 'fn', 'precision', 'recall', f'f{BETA}', 'cost_total_usd']
print('Tabla umbral vs metricas / costo BID:')
display(thresh_df[cols_thresh].round(4))

_t_f2     = best_f2['threshold']
_score_f2 = best_f2[f'f{BETA}']
_cost_f2  = best_f2['cost_total_usd']
_t_cost   = best_cost['threshold']
_cost_min = best_cost['cost_total_usd']
_fp_min   = int(best_cost['fp'])
_fn_min   = int(best_cost['fn'])

print(f'Umbral que maximiza F{BETA}    : {_t_f2:.2f}  (F{BETA}={_score_f2:.4f}, Costo=${_cost_f2:,.0f})')
print(f'Umbral que minimiza Costo BID : {_t_cost:.2f}  (Costo=${_cost_min:,.0f}, FP={_fp_min}, FN={_fn_min})')

Tabla umbral vs metricas / costo BID:


,threshold,fp,fn,precision,recall,f2,cost_total_usd
0,0.15,280,0,0.2689,1.0000,0.6478,378000
1,0.20,257,4,0.2781,0.9612,0.6445,362150
2,0.25,230,8,0.2923,0.9223,0.6445,340900
3,0.30,206,9,0.3133,0.9126,0.6601,312300
4,0.35,178,11,0.3407,0.8932,0.6745,282100
5,0.40,155,17,0.3568,0.8350,0.6585,273850
6,0.45,131,22,0.3821,0.7864,0.6490,260450
7,0.50,103,25,0.4309,0.7573,0.6577,234050
8,0.55,85,35,0.4444,0.6602,0.6018,247750
9,0.60,68,47,0.4516,0.5437,0.5224,270400


Umbral que maximiza F2    : 0.35  (F2=0.6745, Costo=$282,100)
Umbral que minimiza Costo BID : 0.50  (Costo=$234,050, FP=103, FN=25)


## 8. Random Forest — modelo base y tuning

**Random Forest** entrena *B* árboles de decisión en subconjuntos aleatorios de datos y features,
y promedia sus predicciones — reduciendo la varianza sin aumentar el sesgo.

Ventajas relevantes para este problema:
- Robusto a variables irrelevantes gracias al submuestreo de features (`max_features`).
- Maneja desbalance de clases con `class_weight='balanced'`.
- `oob_score=True` estima el error de test sin validación cruzada adicional.

| Prioridad | Hiperparámetros |
|-----------|----------------|
| Alta | `n_estimators` (≈500), `max_features` ([1, √p, p/3, p/2]), `max_depth` |
| Media | `min_samples_leaf`, `min_samples_split`, `class_weight` |

La búsqueda se optimiza sobre **F₂** con `RandomizedSearchCV`.

In [62]:
rf_base = RandomForestClassifier(
    n_estimators=500,
    class_weight='balanced',
    oob_score=True,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf_base.fit(X_train, y_train)

y_prob_rf_base = rf_base.predict_proba(X_test)[:, 1]
y_pred_rf_base = (y_prob_rf_base >= 0.5).astype(int)
metrics_rf_base = evaluate_model(y_test, y_pred_rf_base, y_prob_rf_base)

cols_show = ['accuracy', 'precision', 'recall', 'f1', f'f{BETA}', 'roc_auc', 'cost_total_usd']
print(f'OOB score: {rf_base.oob_score_:.4f}')
print('Métricas — Random Forest base (umbral 0.5):')
display(pd.DataFrame([metrics_rf_base])[cols_show].round(4))

OOB score: 0.8058
Métricas — Random Forest base (umbral 0.5):


,accuracy,precision,recall,f1,f2,roc_auc,cost_total_usd
0,0.7795,0.5455,0.233,0.3265,0.2632,0.757,327200


In [63]:
# Grid según guía del curso (CART y Random Forest, p.31)
rf_distributions = {
    'n_estimators':      [200, 500, 700, 900],
    'max_features':      [1, 'sqrt', 0.33, 0.5], 
    'max_depth':         [None, 10, 20, 30],
    'min_samples_leaf':  [1, 5, 10, 20],
    'min_samples_split': [2, 10, 20],
    'class_weight':      ['balanced', None],
}

random_rf = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=rf_distributions,
    n_iter=N_ITER,
    scoring=SCORING,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=True,
    verbose=0,
)

print('Entrenando RandomizedSearchCV para Random Forest...')
random_rf.fit(X_train, y_train)

print(f'Mejores hiperparámetros : {random_rf.best_params_}')
print(f'Mejor F{BETA} en CV       : {random_rf.best_score_:.4f}')

best_rf    = random_rf.best_estimator_
y_prob_rf  = best_rf.predict_proba(X_test)[:, 1]
y_pred_rf  = (y_prob_rf >= 0.5).astype(int)
metrics_rf = evaluate_model(y_test, y_pred_rf, y_prob_rf)

cols_show = ['accuracy', 'precision', 'recall', 'f1', f'f{BETA}', 'roc_auc', 'cost_total_usd']
print(f'\nMétricas — Random Forest tuneado (umbral 0.5):')
display(pd.DataFrame([metrics_rf])[cols_show].round(4))

Entrenando RandomizedSearchCV para Random Forest...
Mejores hiperparámetros : {'n_estimators': 700, 'min_samples_split': 10, 'min_samples_leaf': 10, 'max_features': 1, 'max_depth': None, 'class_weight': 'balanced'}
Mejor F2 en CV       : 0.6571

Métricas — Random Forest tuneado (umbral 0.5):


,accuracy,precision,recall,f1,f2,roc_auc,cost_total_usd
0,0.7327,0.4485,0.7184,0.5522,0.6412,0.7715,233050


In [64]:
thresholds  = np.arange(0.15, 0.80, 0.05).round(2)
thresh_rf   = evaluate_thresholds(y_test, y_prob_rf, thresholds)

best_f2_rf   = thresh_rf.loc[thresh_rf[f'f{BETA}'].idxmax()]
best_cost_rf = thresh_rf.loc[thresh_rf['cost_total_usd'].idxmin()]

_t_f2_rf     = best_f2_rf['threshold']
_score_f2_rf = best_f2_rf[f'f{BETA}']
_cost_f2_rf  = best_f2_rf['cost_total_usd']
_t_cost_rf   = best_cost_rf['threshold']

print(f'Umbral óptimo F{BETA} : {_t_f2_rf:.2f}  (F{BETA}={_score_f2_rf:.4f}, Costo=${_cost_f2_rf:,.0f})')
print(f'Umbral mínimo costo  : {_t_cost_rf:.2f}  (Costo=${best_cost_rf["cost_total_usd"]:,.0f})')

Umbral óptimo F2 : 0.40  (F2=0.6624, Costo=$303,950)
Umbral mínimo costo  : 0.50  (Costo=$233,050)


## 9. XGBoost — modelo base y tuning

In [65]:
xgb_base = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    learning_rate=0.1,
    n_estimators=300,
    max_depth=5,
    subsample=0.9,
    colsample_bytree=0.9,
    scale_pos_weight=C_FN / C_FP,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
xgb_base.fit(X_train, y_train)

y_prob_xgb_base = xgb_base.predict_proba(X_test)[:, 1]
y_pred_xgb_base = (y_prob_xgb_base >= 0.5).astype(int)
metrics_xgb_base = evaluate_model(y_test, y_pred_xgb_base, y_prob_xgb_base)

cols_show = ['accuracy', 'precision', 'recall', 'f1', f'f{BETA}', 'roc_auc', 'cost_total_usd']
print('Métricas — XGBoost base (umbral 0.5):')
display(pd.DataFrame([metrics_xgb_base])[cols_show].round(4))

Métricas — XGBoost base (umbral 0.5):


,accuracy,precision,recall,f1,f2,roc_auc,cost_total_usd
0,0.7528,0.4565,0.4078,0.4308,0.4167,0.7478,299300


In [ ]:
xgb_distributions = {
    'n_estimators':      [200, 400, 600, 800],   
    'max_depth':         [3, 5, 7],            
    'subsample':         [0.7, 0.8, 0.9],         
    'colsample_bytree':  [0.7, 0.8, 0.9],         
    'reg_lambda':        [0.5, 1.0, 2.0],    
    'reg_alpha':         [0.0, 0.1, 0.5],
    'scale_pos_weight':  [1, C_FN / C_FP, (C_FN / C_FP) ** 0.5],
}

random_xgb = RandomizedSearchCV(
    estimator=XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        learning_rate=0.1,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    param_distributions=xgb_distributions,
    n_iter=N_ITER,
    scoring=SCORING,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=True,
    verbose=0,
)

print('Entrenando RandomizedSearchCV para XGBoost...')
random_xgb.fit(X_train, y_train)

print(f'Mejores hiperparámetros : {random_xgb.best_params_}')
print(f'Mejor F{BETA} en CV       : {random_xgb.best_score_:.4f}')

best_xgb    = random_xgb.best_estimator_
y_prob_xgb  = best_xgb.predict_proba(X_test)[:, 1]
y_pred_xgb  = (y_prob_xgb >= 0.5).astype(int)
metrics_xgb = evaluate_model(y_test, y_pred_xgb, y_prob_xgb)

cols_show = ['accuracy', 'precision', 'recall', 'f1', f'f{BETA}', 'roc_auc', 'cost_total_usd']
print(f'\nMétricas — XGBoost tuneado (umbral 0.5):')
display(pd.DataFrame([metrics_xgb])[cols_show].round(4))

Entrenando RandomizedSearchCV para XGBoost...
Mejores hiperparámetros : {'subsample': 0.9, 'scale_pos_weight': 2.814814814814815, 'reg_lambda': 2.0, 'reg_alpha': 0.5, 'n_estimators': 600, 'max_depth': 3, 'colsample_bytree': 0.8}
Mejor F2 en CV       : 0.5446

Métricas — XGBoost tuneado (umbral 0.5):


,accuracy,precision,recall,f1,f2,roc_auc,cost_total_usd
0,0.7416,0.4404,0.466,0.4528,0.4607,0.7576,291350


In [67]:
thresholds   = np.arange(0.15, 0.80, 0.05).round(2)
thresh_xgb   = evaluate_thresholds(y_test, y_prob_xgb, thresholds)

best_f2_xgb   = thresh_xgb.loc[thresh_xgb[f'f{BETA}'].idxmax()]
best_cost_xgb = thresh_xgb.loc[thresh_xgb['cost_total_usd'].idxmin()]

_t_f2_xgb     = best_f2_xgb['threshold']
_score_f2_xgb = best_f2_xgb[f'f{BETA}']
_cost_f2_xgb  = best_f2_xgb['cost_total_usd']
_t_cost_xgb   = best_cost_xgb['threshold']

print(f'Umbral óptimo F{BETA} : {_t_f2_xgb:.2f}  (F{BETA}={_score_f2_xgb:.4f}, Costo=${_cost_f2_xgb:,.0f})')
print(f'Umbral mínimo costo  : {_t_cost_xgb:.2f}  (Costo=${best_cost_xgb["cost_total_usd"]:,.0f})')

Umbral óptimo F2 : 0.15  (F2=0.6538, Costo=$274,950)
Umbral mínimo costo  : 0.20  (Costo=$265,350)


## 10. Comparación de modelos

Se comparan los tres modelos tuneados evaluados con su umbral óptimo de F₂ en test.

In [68]:
# ── Evaluar cada modelo en su umbral óptimo F₂ ────────────────────────────
def _metrics_at_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    return evaluate_model(y_true, y_pred, y_prob)

m_logit_opt = _metrics_at_threshold(y_test, y_prob_tuned, _t_f2)
m_rf_opt    = _metrics_at_threshold(y_test, y_prob_rf,    _t_f2_rf)
m_xgb_opt   = _metrics_at_threshold(y_test, y_prob_xgb,   _t_f2_xgb)

comp_cols = ['accuracy', 'precision', 'recall', 'f1', f'f{BETA}', 'roc_auc', 'cost_total_usd']
comparacion = pd.DataFrame([
    {'modelo': 'Logit tuneado',    'umbral_optimo': _t_f2,    **{k: m_logit_opt[k] for k in comp_cols}},
    {'modelo': 'Random Forest',    'umbral_optimo': _t_f2_rf,  **{k: m_rf_opt[k]    for k in comp_cols}},
    {'modelo': 'XGBoost',          'umbral_optimo': _t_f2_xgb, **{k: m_xgb_opt[k]   for k in comp_cols}},
]).set_index('modelo').sort_values(f'f{BETA}', ascending=False)

print(f'Comparación de modelos tuneados — evaluados con umbral óptimo F{BETA}:')
display(comparacion.round(4))

# Identificar el mejor modelo por F₂
_mejor_nombre = comparacion[f'f{BETA}'].idxmax()
_mejor_f2     = comparacion.loc[_mejor_nombre, f'f{BETA}']
_mejor_umbral = comparacion.loc[_mejor_nombre, 'umbral_optimo']
print(f'\nMejor modelo : {_mejor_nombre}  (F{BETA}={_mejor_f2:.4f}, umbral={_mejor_umbral:.2f})')

Comparación de modelos tuneados — evaluados con umbral óptimo F2:


,umbral_optimo,accuracy,precision,recall,f1,f2,roc_auc,cost_total_usd
modelo,,,,,,,,
Logit tuneado,0.35,0.5791,0.3407,0.8932,0.4933,0.6745,0.7749,282100
Random Forest,0.40,0.5390,0.3207,0.9029,0.4733,0.6624,0.7715,303950
XGBoost,0.15,0.6192,0.3571,0.8252,0.4985,0.6538,0.7576,274950



Mejor modelo : Logit tuneado  (F2=0.6745, umbral=0.35)


## 11. Exportación del modelo

In [69]:
# ── Seleccionar el mejor modelo por F₂ en umbral óptimo ──────────────────
_candidatos = {
    'Logit tuneado': (best_logit, y_prob_tuned, _t_f2,    m_logit_opt),
    'Random Forest': (best_rf,    y_prob_rf,    _t_f2_rf,  m_rf_opt),
    'XGBoost':       (best_xgb,   y_prob_xgb,   _t_f2_xgb, m_xgb_opt),
}

_nombre_ganador = max(_candidatos, key=lambda k: _candidatos[k][3][f'f{BETA}'])
_modelo_final, _, _umbral_final, _metricas_finales = _candidatos[_nombre_ganador]
_umbral_final = float(_umbral_final)

print(f'Modelo exportado  : {_nombre_ganador}')
print(f'Umbral final      : {_umbral_final}')
print(f'F{BETA} en test    : {_metricas_finales[f"f{BETA}"]:.4f}')
print(f'Recall en test    : {_metricas_finales["recall"]:.4f}')
print(f'Costo total (USD) : ${_metricas_finales["cost_total_usd"]:,.0f}')

# ── Guardar ───────────────────────────────────────────────────────────────
export = {
    'modelo':        _modelo_final,
    'nombre':        _nombre_ganador,
    'umbral':        _umbral_final,
    'features':      list(X_train.columns),
    'beta':          BETA,
    'metricas_test': {k: round(v, 4)
                      for k, v in _metricas_finales.items()
                      if k not in ('tn', 'fp', 'fn', 'tp')},
}

model_path = 'Punto1_modelo.pkl'
os.makedirs('Modelos', exist_ok=True)
joblib.dump(export, os.path.join('Modelos', model_path))
print(f'\nGuardado en : Modelos/{model_path}')

Modelo exportado  : Logit tuneado
Umbral final      : 0.35
F2 en test    : 0.6745
Recall en test    : 0.8932
Costo total (USD) : $282,100

Guardado en : Modelos/Punto1_modelo.pkl
